# 第30章 箱线图（boxplot）

使用中位数、四分位距和须快速比较分布并标记潜在离群点。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

比较一个或多个数值样本的中心、离散程度与异常观察。

## 数据结构

一维样本或多个等价样本列表；类别比较时组名应明确。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 添加 showmeans=True 参数，观察均值标记与中位数线的位置差异
2. 将 whis 参数从默认 1.5 改为 2.0，说明须范围对异常点判定的影响
3. 添加 notch=True 参数，对比缺口箱线图与标准箱线图的视觉效果


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from js import window
base_url = window.location.origin
transactions = pd.read_csv(f"{base_url}/datasets/uci_online_retail_200k.csv", parse_dates=["InvoiceDate"])
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]
transactions["month"] = transactions["InvoiceDate"].dt.to_period("M").astype("string")
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
monthly_summary = completed.groupby("month").agg(sales=("amount", "sum"), orders=("InvoiceNo", "nunique"))
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
profit = sales * 0.18
top_countries = completed.groupby("Country")["amount"].sum().nlargest(4).index
country_rows = transactions[transactions["Country"].isin(top_countries)].copy()
country_rows["flow"] = np.where(country_rows["Quantity"] > 0, "销售", "退货")
country_rows["amount_abs"] = country_rows["amount"].abs()
regional_summary = country_rows.pivot_table(index="Country", columns="flow", values="amount_abs", aggfunc="sum", fill_value=0) / 10_000
regions = regional_summary.index.to_numpy()
online = regional_summary.get("销售", pd.Series(0, index=regional_summary.index)).to_numpy()
offline = regional_summary.get("退货", pd.Series(0, index=regional_summary.index)).to_numpy()
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"UCI Online Retail：{len(transactions):,} 行；图表使用聚合结果与固定样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.boxplot(samples, patch_artist=True, boxprops={"facecolor": "#d2e3fc"}, medianprops={"color": "#d93025", "linewidth": 2})
ax.set(title="订单金额箱线图", ylabel="订单金额（元）", xticks=[1], xticklabels=["全部订单"])
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
rng_box = np.random.default_rng(30)
groups = [
    rng_box.normal(180, 30, 100),
    rng_box.normal(240, 48, 100),
    rng_box.normal(210, 36, 100),
]
fig, ax = plt.subplots(figsize=(8, 4.5))
boxes = ax.boxplot(groups, tick_labels=["办公", "数码", "家居"], patch_artist=True, showmeans=True)
for patch, color in zip(boxes["boxes"], ["#d2e3fc", "#ceead6", "#feefc3"]):
    patch.set_facecolor(color)
ax.set(title="品类订单金额分布", ylabel="订单金额（元）")
fig.tight_layout()
plt.show()


## 3. 参数说明

- whis：须范围
- showmeans：均值
- notch：中位数缺口
- patch_artist：填充箱体


## 4. 结果解读

箱体中线是中位数，箱体覆盖中间50%，须外点是潜在异常而非必然错误。


## 常见误区

- 把箱体高度理解为样本量
- 机械删除所有须外点
- 小样本仍只看箱线摘要


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
rng_delivery = np.random.default_rng(300)
delivery = [rng_delivery.normal(2.6, 0.6, 90), rng_delivery.normal(3.4, 0.9, 90)]
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.boxplot(delivery, tick_labels=["自营", "第三方"], patch_artist=True)
ax.axhline(3, color="#d93025", linestyle="--", label="3天目标")
ax.set(title="配送时长对比", ylabel="天")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## 本章小结

使用中位数、四分位距和须快速比较分布并标记潜在离群点。


### 你已经掌握

- 判断箱线图（boxplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 比较一个或多个数值样本的中心、离散程度与异常观察。 |
| 数据结构 | 一维样本或多个等价样本列表；类别比较时组名应明确。 |
| 结果解读 | 箱体中线是中位数，箱体覆盖中间50%，须外点是潜在异常而非必然错误。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `whis` | 须范围 |
| `showmeans` | 均值 |
| `notch` | 中位数缺口 |
| `patch_artist` | 填充箱体 |


### 需要注意

- 把箱体高度理解为样本量
- 机械删除所有须外点
- 小样本仍只看箱线摘要


### 完成检查

- [ ] 能判断什么问题适合使用箱线图（boxplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
